# 04 — Training and Forecasting

**Purpose:** load the fitted model artifacts, generate current lifecycle forecasts, build the 14-day → 7-day holdout comparison, and export stakeholder-facing outputs. Training logic itself remains in the modeling module.


In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Project root:", PROJECT_ROOT)

Project root: C:\Users\BaneleS\Downloads\Game Performance Model


In [2]:
import pandas as pd

from game_performance.config import SETTINGS
from game_performance.io import load_artifact, load_dataframe, save_dataframe
from game_performance.reporting import build_holdout_comparison, export_excel
from game_performance.scoring import run_daily_lifecycle_scoring

SETTINGS.ensure_directories()

df = load_dataframe(SETTINGS.processed_dir / "clean_game_daily.pkl")
training_results = load_artifact(SETTINGS.model_dir / "training_results.joblib")
model_bundles = training_results["model_bundles"]

print("Loaded cleaned data:", df.shape)
print("Loaded model families:", len(model_bundles))


Loaded cleaned data: (880049, 13)
Loaded model families: 1


In [3]:
current_predictions, scoring_log = run_daily_lifecycle_scoring(
    df,
    model_bundles,
    SETTINGS.horizon_strategy,
    as_of_date=df["Date"].max(),
    exact_checkpoint_only=False,
)

print(f"Generated {len(current_predictions):,} predictions")
display(current_predictions.head(30))

if scoring_log is not None and len(scoring_log): 
    print("Scoring log")
    display(scoring_log)


Generated 12,491 predictions


,Game_ID,TournamentName,SystemName,First_Observed_Date,Lifecycle_Stage,Obs_Window,Horizon,active_observation_days,observation_coverage,predicted_clients_week_4,...,predicted_stakes_week_4,confidence_stakes,raw_probability_high_performance,probability_high_performance,classification_threshold,predicted_high_performer,recommendation,scored_at,Checkpoint,As_Of_Date
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,AmusnetS-100 Bulky Fruits,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,418.933014,...,2.341061e+05,0.631029,0.065084,0.000000,0.5,0,Standard catalogue placement,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
1,AmusnetSlots::AmusnetS-100 Cats,AmusnetS-100 Cats,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,10081.802087,...,2.624346e+06,0.850654,0.988752,1.000000,0.5,1,Priority placement for week 4,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
2,AmusnetSlots::AmusnetS-100 Power Hot,AmusnetS-100 Power Hot,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,9792.226600,...,3.068016e+06,0.850654,0.988752,1.000000,0.5,1,Priority placement for week 4,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
3,AmusnetSlots::AmusnetS-2 Dragons,AmusnetS-2 Dragons,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,704.209100,...,1.868011e+05,0.631029,0.073895,0.000000,0.5,0,Standard catalogue placement,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,AmusnetS-20 Bulky Fruits,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,3766.676646,...,1.051917e+06,0.509886,0.965415,1.000000,0.5,1,Priority placement for week 4,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
5,AmusnetSlots::AmusnetS-20 Diamonds,AmusnetS-20 Diamonds,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,662.424272,...,1.990007e+05,0.631029,0.090340,0.095238,0.5,0,Standard catalogue placement,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
6,AmusnetSlots::AmusnetS-20 Golden Coins,AmusnetS-20 Golden Coins,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,976.859877,...,3.141303e+05,0.631029,0.202917,0.095238,0.5,0,Standard catalogue placement,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
7,AmusnetSlots::AmusnetS-20 Hot Blast,AmusnetS-20 Hot Blast,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,7813.057453,...,1.626406e+06,0.850654,0.986442,1.000000,0.5,1,Priority placement for week 4,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
8,AmusnetSlots::AmusnetS-20 Joker Reels,AmusnetS-20 Joker Reels,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,6725.914761,...,1.559163e+06,0.850654,0.986760,1.000000,0.5,1,Priority placement for week 4,2026-09-03 14:26:51.108695+00:00,14,2026-07-29
9,AmusnetSlots::AmusnetS-20 Power Hot,AmusnetS-20 Power Hot,AmusnetSlots,2026-02-01,Week 2,14,14,14,1.0,5301.996345,...,1.042240e+06,0.509886,0.980423,1.000000,0.5,1,Priority placement for week 4,2026-09-03 14:26:51.108695+00:00,14,2026-07-29


Scoring log


,Game_ID,Checkpoint,Horizon,As_Of_Date,scored_at
0,AmusnetSlots::AmusnetS-100 Bulky Fruits,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
1,AmusnetSlots::AmusnetS-100 Cats,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
2,AmusnetSlots::AmusnetS-100 Power Hot,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
3,AmusnetSlots::AmusnetS-2 Dragons,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
4,AmusnetSlots::AmusnetS-20 Bulky Fruits,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
...,...,...,...,...,...
12486,YGG::YGG-Goals of Olympus BoltLock,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
12487,YGG::YGG-Lucky Coin Festival,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
12488,YGG::YGG-Sidewinder Xtra Power,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00
12489,YGG::YGG-The Good Farmer,14,14,2026-07-29,2026-09-03 14:26:51.108695+00:00


In [4]:
holdout = training_results["holdout_predictions"]

print("Holdout predictions:", holdout.shape)

if holdout.empty:
    raise ValueError(
        "No holdout predictions are available. Run 03_model_experiments_production.ipynb first."
    )

available = (
    holdout[["Obs_Window", "Horizon"]]
    .drop_duplicates()
    .sort_values(["Obs_Window", "Horizon"])
    .reset_index(drop=True)
)

print("Available holdout model families:")
display(available)

available_pairs = {
    (int(row.Obs_Window), int(row.Horizon))
    for row in available.itertuples(index=False)
}

# Production preference. If full production training was run, this should exist.
preferred_pair = (14, 7)

if preferred_pair in available_pairs:
    checkpoint, horizon = preferred_pair
else:
    # Keep the notebook runnable, but make the issue obvious.
    checkpoint, horizon = tuple(available.iloc[-1].astype(int))
    print(
        f"WARNING: preferred production pair {preferred_pair} is not available. "
        f"Using checkpoint={checkpoint}, horizon={horizon} instead. "
        "Check skipped_configurations from notebook 03."
    )

comparison = build_holdout_comparison(
    holdout,
    checkpoint=int(checkpoint),
    horizon=int(horizon),
)

print(f"Comparison rows: {len(comparison):,}")
display(comparison.head(20))


Holdout predictions: (498, 28)
Available holdout model families:


,Obs_Window,Horizon
0,14,14


Comparison rows: 498


,Game_ID,TournamentName,SystemName,First_Observed_Date,Observation_Start,Observation_End,Prediction_Start,Prediction_End,Obs_Window,Horizon,...,predicted_stakes_week_4,confidence_stakes,variance_stakes,probability_stakes,actual_clients,actual_clients_week_1,actual_clients_week_2,actual_stakes,actual_stakes_week_1,actual_stakes_week_2
0,SyX-Online::Iceland League U20 Women,Iceland League U20 Women,SyX-Online,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,1.113193e+04,0.015575,-60459.112578,0.155493,5186,<NA>,5186,71591.04,<NA>,71591.04
1,Colossus::COL-WC,COL-WC,Colossus,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,7.655642e+03,0.109027,-10706.307857,0.416930,564,530,564,18361.95,16636.24,18361.95
2,Erlang-Online::Iceland League U20 Women,Iceland League U20 Women,Erlang-Online,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,NaN,NaN,<NA>,NaN,<NA>,NaN,NaN,<NA>,NaN,NaN
3,SplitThePot::STP-daredevil,STP-daredevil,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,1.699300e+05,0.599569,64920.285861,0.381769,266,906,266,105009.68,246394.53,105009.68
4,SplitThePot::STP-paperplane,STP-paperplane,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,1.099695e+05,0.599569,77392.06281,0.000000,192,583,192,32577.46,654431.99,32577.46
5,SplitThePot::STP-Rugby Run Twins,STP-Rugby Run Twins,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,4.913477e+05,0.553470,-85918.749019,0.851163,1045,2811,1045,577266.48,1433806.98,577266.48
6,SplitThePot::STP-Zama Fruits,STP-Zama Fruits,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,3.517658e+06,0.850654,1083234.277121,0.555035,9336,33150,9336,2434423.63,6595433.69,2434423.63
7,SplitThePot::STP-Zama Fruits-Data Free,STP-Zama Fruits-Data Free,SplitThePot,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,9.357859e+05,0.509886,35312.191424,0.960785,6797,18450,6797,900473.73,2354337.11,900473.73
8,SplitThePot Free Spins::STP-Zama Fruits,STP-Zama Fruits,SplitThePot Free Spins,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,8.616345e+01,0.080934,86.163446,0.000000,17,45,17,0.0,0.0,0.0
9,SplitThePot Free Spins::STP-Zama Fruits-Data Free,STP-Zama Fruits-Data Free,SplitThePot Free Spins,2026-06-08,2026-06-08,2026-06-21,2026-06-22,2026-07-05,14,14,...,0.000000e+00,0.080934,0.0,1.000000,25,60,25,0.0,0.0,0.0


In [5]:
from datetime import datetime
from pathlib import Path


def export_production_excel(
    path: Path,
    *,
    comparison,
    forecasts,
    regression,
    classification,
) -> Path:
    """Export the production report, using a timestamped copy if Excel has it open."""
    path.parent.mkdir(parents=True, exist_ok=True)

    try:
        with open(path, "a"):
            pass
    except PermissionError:
        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        path = path.with_name(f"{path.stem}_{timestamp}{path.suffix}")
        print("Report is currently open.\n" f"Saving instead to:\n{path}")

    return export_excel(
        path,
        comparison=comparison,
        forecasts=forecasts,
        regression=regression,
        classification=classification,
    )


In [6]:
save_dataframe(current_predictions, SETTINGS.report_dir / "current_predictions.pkl")

report_path = SETTINGS.report_dir / "model_business_report.xlsx"

export_production_excel(
    report_path,
    comparison=comparison,
    forecasts=current_predictions,
    regression=training_results["regression_evaluation"],
    classification=training_results["classification_evaluation"],
)

print("Saved current predictions:", SETTINGS.report_dir / "current_predictions.pkl")
print("Saved Excel report:", report_path)


Report is currently open.
Saving instead to:
C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\reports\model_business_report_20260903_162651.xlsx
Saved current predictions: C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\reports\current_predictions.pkl
Saved Excel report: C:\Users\BaneleS\Downloads\Game Performance Model\artifacts\reports\model_business_report.xlsx
